In [134]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn import linear_model, svm
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
model_df = pd.read_excel('valmis_dataset.xlsx')

In [14]:
model_df = model_df.iloc[:, 2:-1]

In [15]:
X = model_df.iloc[:, :10]
y = model_df.iloc[:, 10:-1]

In [87]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [19]:
# dummy regression, tekee predit yksinkertaisilla säännöillä

dummy = DummyRegressor(strategy='median')
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)
mae = mean_absolute_error(y_test, dummy_preds)
print(f'mediaani baseline: {mae}')

# ekan kierroksen dummy regressor mae: 7208.749382473783

mediaani baseline: 6041.926529768914


In [ ]:
"""
RandomForsetRegressor
Parametrituunaus X_train, y_train -> 
cross-va X_test, y_test ->
"""

param_grid = {
    'n_estimators': [100,200,300,400,500,600,700],
    'min_samples_leaf': [1,2,3,4,5],
}
model = RandomForestRegressor(random_state=42)

grid = GridSearchCV(
    model,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)
grid.best_params_

{'min_samples_leaf': 1, 'n_estimators': 300}

In [94]:
preds = grid.best_estimator_.predict(X_test)
mean_absolute_error(y_test, preds)

2852.1982845880552

In [101]:
#reg.get_params()

In [102]:
param_grid = {
    'ridge__alpha': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
}

reg = make_pipeline(
    StandardScaler(),
    linear_model.Ridge(random_state=42)
)

grid = GridSearchCV(
    reg,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)
grid.best_params_

{'ridge__alpha': 1.0}

In [103]:
preds = grid.best_estimator_.predict(X_test)
mean_absolute_error(y_test, preds)

3029.279070254851

In [104]:
# MAE per luokittelu

err = np.abs(y_test.values - preds)

mae_by_target = pd.Series(
    err.mean(axis=0),
    index=y_test.columns
).sort_values(ascending=False)

mae_by_target

työmaan yleiskulut      6527.075932
purkutyöt               5501.871774
lattian_asennukset      3705.675064
maalaukset_y            1736.195068
ovet_ja_listoitukset    1464.513263
laatoitukset            1319.545299
alakatot                 950.077091
dtype: float64

In [106]:
'''
mean_target: kategorian keskiarvo
mae: mean average error

purkutyöt, ovet ja listotukset sekä yleiskulut errorit suhteettman suuria keskiarvoon.
'''

target_summary = pd.DataFrame({
    'mae': mae_by_target,
    'mean_target': y_test.mean(),
    'mae_pct_of_mean': mae_by_target / y_test.mean() * 100
}).sort_values('mae_pct_of_mean', ascending=False)

target_summary



,mae,mean_target,mae_pct_of_mean
ovet_ja_listoitukset,1464.513263,2499.437138,58.593723
purkutyöt,5501.871774,9509.944891,57.853877
laatoitukset,1319.545299,3212.280976,41.078141
työmaan yleiskulut,6527.075932,17594.203630,37.097876
lattian_asennukset,3705.675064,13972.421246,26.521352
alakatot,950.077091,5346.509149,17.770045
maalaukset_y,1736.195068,13037.182383,13.317257


In [ ]:
# Kertoimet

ridge = reg.named_steps['ridge']
coef_df = pd.DataFrame(
    ridge.coef_, 
    index=y_train.columns,
    columns=X_train.columns
)

coef_df

In [108]:
np.linspace(0.1, 1, 10)

array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])

In [ ]:
# ridge ilman standard scaleria

reg = linear_model.Ridge(random_state=42)

param_grid = {
    'alpha': np.linspace(0.1, 1, 10)
}

grid = GridSearchCV(
    reg,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)
grid.best_params_

{'alpha': np.float64(1.0)}

In [110]:
preds = grid.best_estimator_.predict(X_test)
mean_absolute_error(y_test, preds)

3032.7656636787547

In [111]:
err = np.abs(y_test.values - preds)

mae_by_target = pd.Series(
    err.mean(axis=0),
    index=y_test.columns
).sort_values(ascending=False)

mae_by_target

työmaan yleiskulut      6520.764781
purkutyöt               5505.277945
lattian_asennukset      3700.607176
maalaukset_y            1788.747590
ovet_ja_listoitukset    1458.398756
laatoitukset            1324.152387
alakatot                 931.411011
dtype: float64

In [112]:
target_summary = pd.DataFrame({
    'mae': mae_by_target,
    'mean_target': y_test.mean(),
    'mae_pct_of_mean': mae_by_target / y_test.mean() * 100
}).sort_values('mae_pct_of_mean', ascending=False)

target_summary

,mae,mean_target,mae_pct_of_mean
ovet_ja_listoitukset,1458.398756,2499.437138,58.349087
purkutyöt,5505.277945,9509.944891,57.889693
laatoitukset,1324.152387,3212.280976,41.221562
työmaan yleiskulut,6520.764781,17594.203630,37.062006
lattian_asennukset,3700.607176,13972.421246,26.485082
alakatot,931.411011,5346.509149,17.420919
maalaukset_y,1788.747590,13037.182383,13.720354


In [ ]:
pd.DataFrame(
    reg.coef_,
    index=y_train.columns,
    columns=X_train.columns
)

In [116]:
# GradientBoostingRegressor

base = GradientBoostingRegressor(random_state=42)
model = MultiOutputRegressor(base)

param_grid = {
    'estimator__learning_rate': [0.03, 0.05, 0.1, 0.2, 0.3],
    'estimator__n_estimators': [100, 200, 300, 400, 500, 600, 700],
    'estimator__max_depth': [2, 3],
}

grid = GridSearchCV(
    model,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=1
)

grid.fit(X_train, y_train)
grid.best_params_

{'estimator__learning_rate': 0.1,
 'estimator__max_depth': 3,
 'estimator__n_estimators': 700}

In [118]:
preds = grid.best_estimator_.predict(X_test)
mean_absolute_error(y_test, preds)

2196.3029905083927

In [119]:
print('Best CV MAE:', -grid.best_score_)
print('Final test MAE:', mean_absolute_error(y_test, preds))

Best CV MAE: 3175.3898983433464
Final test MAE: 2196.3029905083927


In [32]:
reg = linear_model.LinearRegression()
reg.fit(X_train, y_train)

preds = reg.predict(X_test)

mse = mean_absolute_error(y_test, preds)

mse

3032.1090177062288

array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. , 1.1, 1.2, 1.3,
       1.4, 1.5])

In [ ]:
# elastic net

model = linear_model.ElasticNet(random_state=42)

param_grid = {
    'alpha': np.linspace(0.1, 1.5, 15),
    'l1_ratio': np.linspace(0.1, 1, 10)
}

grid = GridSearchCV(
    model,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)
grid.best_params_

In [131]:
preds = grid.best_estimator_.predict(X_test)
print('Best CV MAE:', -grid.best_score_)
print('Final test MAE:', mean_absolute_error(y_test, preds))

Best CV MAE: 3483.0284452491446
Final test MAE: 3052.507662363306


In [132]:
err = np.abs(y_test.values - preds)

mae_by_target = pd.Series(
    err.mean(axis=0),
    index=y_test.columns
).sort_values(ascending=False)

mae_by_target

työmaan yleiskulut      6525.847802
purkutyöt               5644.470485
lattian_asennukset      3660.637455
maalaukset_y            1768.960187
ovet_ja_listoitukset    1479.020603
laatoitukset            1354.493070
alakatot                 934.124034
dtype: float64

In [133]:
target_summary = pd.DataFrame({
    'mae': mae_by_target,
    'mean_target': y_test.mean(),
    'mae_pct_of_mean': mae_by_target / y_test.mean() * 100
}).sort_values('mae_pct_of_mean', ascending=False)

target_summary

,mae,mean_target,mae_pct_of_mean
purkutyöt,5644.470485,9509.944891,59.353346
ovet_ja_listoitukset,1479.020603,2499.437138,59.174147
laatoitukset,1354.493070,3212.280976,42.166083
työmaan yleiskulut,6525.847802,17594.203630,37.090896
lattian_asennukset,3660.637455,13972.421246,26.199020
alakatot,934.124034,5346.509149,17.471663
maalaukset_y,1768.960187,13037.182383,13.568577


In [135]:
base = HistGradientBoostingRegressor(random_state=42)

model = MultiOutputRegressor(base)

param_grid = {
    "estimator__learning_rate": [0.03, 0.05, 0.1],
    "estimator__max_iter": [100, 200, 300],
    "estimator__max_leaf_nodes": [15, 31, 63],
    "estimator__l2_regularization": [0.0, 0.1, 1.0],
}

grid = GridSearchCV(
    model,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

preds = grid.best_estimator_.predict(X_test)
mean_absolute_error(y_test, preds)

3369.016300644901